Importing deepstack for powerful ensembles, [more info](https://github.com/jcborges/DeepStack)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd /content/drive/MyDrive/Stroke Classification

/content/drive/MyDrive/Stroke Classification


In [ ]:
!pip install tensorflow==2.10.0

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
!pip install deepstack

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


**Importing essential pacakages**

In [ ]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.layers import Flatten,Dense,Dropout,BatchNormalization
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.layers import PReLU
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras import layers

Specifying global parameters

In [ ]:
img_h,img_w= (224,224)
batch_size=32
epochs=25
n_class=2

*Concatenating train and test directory paths..*

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(validation_split=0.15)
valid_datagen= ImageDataGenerator(validation_split=0.15)
test_datagen= ImageDataGenerator()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_generator = train_datagen.flow_from_directory(
                    "Brain_Data_Organised/Train",   # This is the source directory for training images
                    target_size=(img_h, img_w),  #
                    batch_size=batch_size,
                    subset='training',
                    seed=1337,
                    class_mode='categorical')

validation_generator = train_datagen.flow_from_directory(
                        "Brain_Data_Organised/Train",
                        target_size=(img_h, img_w),
                        batch_size=batch_size,
                        subset='validation',
                        seed=333,
                        class_mode='categorical')

test_generator = test_datagen.flow_from_directory(
                        "Brain_Data_Organised/Test",
                        target_size=(img_h, img_w),
                        batch_size=batch_size,
                        class_mode='categorical',
                        shuffle=False)

Found 2021 images belonging to 2 classes.
Found 355 images belonging to 2 classes.
Found 250 images belonging to 2 classes.


Importing and initializing VGG-19

In [ ]:
from tensorflow.keras.applications.efficientnet import EfficientNetB2, preprocess_input
from tensorflow.keras.layers import Dense,Conv2D, Flatten, Dropout, MaxPooling2D, GlobalAveragePooling2D

base_model_1=tf.keras.applications.EfficientNetB2(include_top=False, weights='imagenet',input_shape=(img_h,img_w,3))

# Making last layers trainable, because our dataset is much diiferent from the imagenet dataset
for layer in base_model_1.layers:
    layer.trainable=True

model_1=Sequential()
model_1.add(base_model_1)
model_1.add(GlobalAveragePooling2D())
model_1.add(Dense(2,activation='softmax'))


9406464/9406464 [==============================] - 0s 0us/step


Plotting Vgg-19 model

In [ ]:
from tensorflow.keras.applications.densenet import DenseNet121

base_model_2= DenseNet121(include_top=False, weights='imagenet',
                                         input_shape=(img_h,img_w,3))

for layer in base_model_2.layers:
    layer.trainable=True

model_2=Sequential()
model_2.add(base_model_2)
model_2.add(GlobalAveragePooling2D())
model_2.add(Dense(2,activation='softmax'))

29084464/29084464 [==============================] - 0s 0us/step


In [ ]:
from tensorflow.keras.applications.xception import Xception

base_model_3= Xception(include_top=False, weights='imagenet',
                                        input_shape=(img_h,img_w,3))

for layer in base_model_3.layers:
    layer.trainable=True

model_3=Sequential()
model_3.add(base_model_3)
model_3.add(GlobalAveragePooling2D())
model_3.add(Dense(2,activation='softmax'))

83683744/83683744 [==============================] - 3s 0us/step


Plotting Inception_V3 model

Creating callbacks and optimizers. You may try out diiferent optimizers

In [ ]:
from tensorflow.keras.optimizers import Adam,SGD,Adagrad,Adadelta,RMSprop

reduce_learning_rate = ReduceLROnPlateau(monitor='loss',
                                         factor=0.1,
                                         patience=3,
                                         cooldown=2,
                                         min_lr=1e-10,
                                         verbose=1)

callbacks = [reduce_learning_rate]
optimizer = Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999)

In [ ]:
def get_callbacks(model_name):
    callbacks =[]
    checkpoint = tf.keras.callbacks.ModelCheckpoint(filepath=f'model.{model_name}.keras', verbose=1, monitor='val_loss',mode='min',save_best_only=True)
    callbacks.append(checkpoint)
    anne = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=2, min_lr=0.0000001,min_delta=0.00001,mode='auto')
    callbacks.append(anne)
    earlystop = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=8)
    callbacks.append(earlystop)
    return callbacks

Compiling Models

In [ ]:
#model_1.compile( loss='categorical_crossentropy',optimizer= optimizer, metrics=['accuracy',keras.metrics.Precision(), keras.metrics.Recall(), keras.metrics.SpecificityAtSensitivity(0.5), keras.metrics.SensitivityAtSpecificity(0.5)])
model_2.compile( loss='categorical_crossentropy',optimizer= optimizer, metrics=['accuracy',keras.metrics.Precision(), keras.metrics.Recall(), keras.metrics.SpecificityAtSensitivity(0.5), keras.metrics.SensitivityAtSpecificity(0.5)])
#model_3.compile( loss='categorical_crossentropy',optimizer= optimizer, metrics=['accuracy',keras.metrics.Precision(), keras.metrics.Recall(), keras.metrics.SpecificityAtSensitivity(0.5), keras.metrics.SensitivityAtSpecificity(0.5)])

Now lets fit the models on our dataset

In [ ]:
history_1 = model_1.fit(
      train_generator,
      epochs=epochs,
      validation_data=validation_generator,
      callbacks=callbacks,
      verbose=1)

Epoch 1/25
67/67 [==============================] - 26s 240ms/step - loss: 0.4763 - accuracy: 0.7873 - precision: 0.7873 - recall: 0.7873 - specificity_at_sensitivity: 0.9388 - sensitivity_at_specificity: 0.9388 - val_loss: 7.0503 - val_accuracy: 0.5992 - val_precision: 0.5992 - val_recall: 0.5992 - val_specificity_at_sensitivity: 0.5992 - val_sensitivity_at_specificity: 0.5992 - lr: 0.0010
Epoch 2/25
67/67 [==============================] - 13s 198ms/step - loss: 0.1918 - accuracy: 0.9280 - precision: 0.9280 - recall: 0.9280 - specificity_at_sensitivity: 0.9953 - sensitivity_at_specificity: 0.9953 - val_loss: 9.1996 - val_accuracy: 0.5992 - val_precision: 0.5992 - val_recall: 0.5992 - val_specificity_at_sensitivity: 0.6160 - val_sensitivity_at_specificity: 0.6160 - lr: 0.0010
Epoch 3/25
67/67 [==============================] - 12s 176ms/step - loss: 0.1447 - accuracy: 0.9439 - precision: 0.9439 - recall: 0.9439 - specificity_at_sensitivity: 0.9981 - sensitivity_at_specificity: 0.9981 

In [ ]:
model_1.save("efficient.keras")

In [ ]:
history_2 = model_2.fit(
      train_generator,
      epochs=10,
      validation_data=validation_generator,
      callbacks=callbacks,
      verbose=1)

Epoch 1/10
64/64 [==============================] - 47s 479ms/step - loss: 0.5632 - accuracy: 0.7264 - precision_1: 0.7264 - recall_1: 0.7264 - specificity_at_sensitivity_1: 0.8931 - sensitivity_at_specificity_1: 0.8931 - val_loss: 4.1671 - val_accuracy: 0.5775 - val_precision_1: 0.5775 - val_recall_1: 0.5775 - val_specificity_at_sensitivity_1: 0.6169 - val_sensitivity_at_specificity_1: 0.6169 - lr: 0.0010
Epoch 2/10
64/64 [==============================] - 26s 401ms/step - loss: 0.2611 - accuracy: 0.8906 - precision_1: 0.8906 - recall_1: 0.8906 - specificity_at_sensitivity_1: 0.9921 - sensitivity_at_specificity_1: 0.9921 - val_loss: 11.2641 - val_accuracy: 0.6901 - val_precision_1: 0.6901 - val_recall_1: 0.6901 - val_specificity_at_sensitivity_1: 0.7352 - val_sensitivity_at_specificity_1: 0.7352 - lr: 0.0010
Epoch 3/10
64/64 [==============================] - 25s 392ms/step - loss: 0.1923 - accuracy: 0.9243 - precision_1: 0.9243 - recall_1: 0.9243 - specificity_at_sensitivity_1: 0.995

In [ ]:
model_2.save("DensNet_1.keras")

In [ ]:
history_3 = model_3.fit(
      train_generator,
      epochs=epochs,
      validation_data=validation_generator,
      callbacks=callbacks,
      verbose=1)

Epoch 1/25
74/74 [==============================] - 58s 670ms/step - loss: 0.6049 - accuracy: 0.6672 - precision_2: 0.6672 - recall_2: 0.6672 - specificity_at_sensitivity_2: 0.8049 - sensitivity_at_specificity_2: 0.8049 - val_loss: 3.3609 - val_accuracy: 0.4821 - val_precision_2: 0.4821 - val_recall_2: 0.4821 - val_specificity_at_sensitivity_2: 0.4777 - val_sensitivity_at_specificity_2: 0.4777 - lr: 0.0010
Epoch 2/25
74/74 [==============================] - 48s 652ms/step - loss: 0.2822 - accuracy: 0.8827 - precision_2: 0.8827 - recall_2: 0.8827 - specificity_at_sensitivity_2: 0.9898 - sensitivity_at_specificity_2: 0.9898 - val_loss: 1.4002 - val_accuracy: 0.7991 - val_precision_2: 0.7991 - val_recall_2: 0.7991 - val_specificity_at_sensitivity_2: 0.8884 - val_sensitivity_at_specificity_2: 0.8884 - lr: 0.0010
Epoch 3/25
74/74 [==============================] - 48s 644ms/step - loss: 0.1603 - accuracy: 0.9411 - precision_2: 0.9411 - recall_2: 0.9411 - specificity_at_sensitivity_2: 0.9966

In [ ]:
model_3.save("Xception.keras")

In [ ]:
from tensorflow import keras
model1 = keras.models.load_model('efficient.keras')
model2 = keras.models.load_model('DensNet_1.keras')
model3 = keras.models.load_model('Xception.keras')

In [ ]:
from deepstack.base import KerasMember

member1 = KerasMember(name="efficientnet", keras_model=model1, train_batches=train_generator, val_batches=test_generator)
member2 = KerasMember(name="DensNet", keras_model=model2, train_batches=train_generator, val_batches=test_generator)
member3 = KerasMember(name="Xception", keras_model=model3, train_batches=train_generator, val_batches=test_generator)

/usr/local/lib/python3.8/dist-packages/deepstack/base.py:147: UserWarning: `Model.predict_generator` is deprecated and will be removed in a future version. Please use `Model.predict`, which supports generators.
  return self.model.predict_generator(


8/8 [==============================] - 3s 324ms/step


In [ ]:
from deepstack.ensemble import DirichletEnsemble
from sklearn.metrics import accuracy_score

wAvgEnsemble = DirichletEnsemble(N=5000, metric=accuracy_score)
wAvgEnsemble.add_members([member1, member2, member3])
wAvgEnsemble.fit()
wAvgEnsemble.describe()

efficientnet - Weight: 0.0230 - accuracy_score: 0.8703
DensNet - Weight: 0.0002 - accuracy_score: 0.9335
Xception - Weight: 0.9768 - accuracy_score: 0.9963
DirichletEnsemble accuracy_score: 0.9984


In [ ]:
wAvgEnsemble.save("Ensemble.keras")